In [ ]:
# --- Imports ---
import ipyvuetify as v
import sepal_ui.sepalwidgets as sui
from sepal_ui import model as sumodel
import traitlets
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from pathlib import Path
import traceback
import random
import os
import seaborn as sns
import pandas as pd
from scipy import stats

print("Loading components for Burn Severity Correlation App (v6)...")

# --- Model ---
class BurnModel(sumodel.Model):
    """Holds the application's data and state."""
    input_raster = traitlets.Unicode(None, allow_none=True).tag(sync=True)
    available_bands = traitlets.List([]).tag(sync=True)
    selected_nbr_band = traitlets.Int(None, allow_none=True).tag(sync=True)
    selected_cov_band = traitlets.Int(None, allow_none=True).tag(sync=True)
    alert_message = traitlets.Unicode("Select a multi-band raster file.").tag(sync=True)
    alert_type = traitlets.Unicode("info").tag(sync=True)
    loading = traitlets.Bool(False).tag(sync=True)
    available_charts = traitlets.List([
        {'text': 'Scatter Plot (with R² & Line)', 'value': 'scatter'},
        {'text': 'Violin Plot (Binned Distribution)', 'value': 'violin'},
    ]).tag(sync=True)
    selected_chart = traitlets.Unicode('scatter').tag(sync=True)
    mask_enabled = traitlets.Bool(True).tag(sync=True)
    mask_value = traitlets.Float(0.0).tag(sync=True)

# --- Backend Functions (Remain the same) ---
def get_raster_bands(filepath_str):
    if not filepath_str or not Path(filepath_str).exists(): raise FileNotFoundError("Raster not found.")
    bands_list = []
    with rasterio.open(filepath_str) as src:
        if src.count < 2: raise ValueError("Raster must have at least two bands.")
        descriptions = src.descriptions
        for i in range(1, src.count + 1):
            band_name = descriptions[i-1] if descriptions and descriptions[i-1] else f"Band {i}"
            bands_list.append({'text': band_name, 'value': i})
    return bands_list

def get_band_data(filepath_str, band1_idx, band2_idx, sample_size=10000,
                  enable_mask=False, value_to_mask=0.0):
    if not filepath_str: raise ValueError("Filepath missing.")
    if not band1_idx or not band2_idx: raise ValueError("Select both bands.")
    if band1_idx == band2_idx: raise ValueError("Bands cannot be the same.")
    with rasterio.open(filepath_str) as src:
        data1 = src.read(band1_idx, masked=True)
        data2 = src.read(band2_idx, masked=True)
        combined_mask = np.logical_or(data1.mask, data2.mask)
        if enable_mask:
            print(f"Applying mask for value: {value_to_mask} on Y-axis band.")
            value_mask = np.logical_and(~combined_mask, (data2 == value_to_mask))
            combined_mask = np.logical_or(combined_mask, value_mask)
        valid_data1 = data1[~combined_mask].flatten()
        valid_data2 = data2[~combined_mask].flatten()
        num_valid = len(valid_data1)
        if num_valid == 0: raise ValueError("No valid data found after masking.")
        indices = random.sample(range(num_valid), min(num_valid, sample_size))
        return valid_data1[indices], valid_data2[indices]

# --- Controller ---
class BurnController:
    """Manages UI elements and interactions."""

    def __init__(self, model: BurnModel):
        self.model = model
        self.plot_output = widgets.Output()
        self.app = self._create_ui()
        self.model.observe(self._update_alert_display, 'alert_message')
        self.model.observe(self._update_alert_type, 'alert_type')
        self.model.observe(self._update_mask_input_disabled, 'mask_enabled')
        print("BurnController initialized.")

    def _update_alert_display(self, change):
        self.alert.children = [change['new']]
        self.alert.show()

    def _update_alert_type(self, change):
        self.alert.type = change['new']

    def _update_mask_input_disabled(self, change):
        self.mask_input.disabled = not change['new']

    def _create_ui(self):
        """Builds the user interface."""
        self.file_input = sui.FileInput(label="Select Raster File", folder=os.getcwd(), extensions=['.tif', '.tiff', '.img', '.vrt'])
        self.nbr_select = sui.Select(label="Select NBR/Y-axis Band", items=self.model.available_bands, v_model=None)
        self.cov_select = sui.Select(label="Select Covariate/X-axis Band", items=self.model.available_bands, v_model=None)
        self.chart_select = sui.Select(label="Select Chart Type", items=self.model.available_charts, v_model=self.model.selected_chart)
        self.mask_check = sui.Checkbox(label="Mask NBR/Y-axis Value?", v_model=self.model.mask_enabled)
        self.mask_input = sui.NumberField(label="Value to Mask", v_model=self.model.mask_value, type='number')
        self.plot_btn = sui.Btn("Generate Plot", icon="mdi-chart-line", class_="ma-2", disabled=True)
        self.alert = sui.Alert(children=[self.model.alert_message], type=self.model.alert_type).show()

        # Linkages
        traitlets.link((self.file_input, 'v_model'), (self.model, 'input_raster'))
        traitlets.link((self.nbr_select, 'items'), (self.model, 'available_bands'))
        traitlets.link((self.cov_select, 'items'), (self.model, 'available_bands'))
        traitlets.link((self.nbr_select, 'v_model'), (self.model, 'selected_nbr_band'))
        traitlets.link((self.cov_select, 'v_model'), (self.model, 'selected_cov_band'))
        widgets.link((self.model, 'loading'), (self.plot_btn, 'loading'))
        traitlets.link((self.chart_select, 'v_model'), (self.model, 'selected_chart'))
        traitlets.link((self.mask_check, 'v_model'), (self.model, 'mask_enabled'))
        traitlets.link((self.mask_input, 'v_model'), (self.model, 'mask_value'))

        # Observers & Events
        self.model.observe(self._check_plot_readiness, names=['selected_nbr_band', 'selected_cov_band'])
        self.model.observe(self._on_file_selected, 'input_raster')
        self.plot_btn.on_event('click', self._on_plot_click)

        self.mask_input.disabled = not self.model.mask_enabled

        # Layout
        app_layout = v.Card(class_="pa-4", children=[
            v.CardTitle(children=["Burn Severity Correlation Mapper"]),
            v.CardText(children=["Select a raster, bands, mask option, and chart type."]),
            self.file_input,
            v.Row(children=[v.Col(cols=6, children=[self.nbr_select]), v.Col(cols=6, children=[self.cov_select])]),
            v.Row(children=[v.Col(cols=6, children=[self.mask_check]), v.Col(cols=6, children=[self.mask_input])]),
            v.Row(children=[v.Col(cols=6, children=[self.chart_select]), v.Col(cols=6, children=[self.plot_btn])]),
            self.alert,
            v.Divider(class_="my-4"),
            v.CardTitle(children=["Plot Output"]),
            self.plot_output,
        ])
        return app_layout

    def _check_plot_readiness(self, change):
        self.plot_btn.disabled = not (self.model.selected_nbr_band and self.model.selected_cov_band)

    def _on_file_selected(self, change):
        filepath = change['new']
        self.model.available_bands = []
        self.model.selected_nbr_band = None
        self.model.selected_cov_band = None
        self.plot_output.clear_output()
        if filepath:
            try:
                self.model.loading = True
                self.model.alert_message = "Reading bands..."
                self.model.alert_type = "info"
                self.model.available_bands = get_raster_bands(filepath)
                self.model.alert_message = "Select bands and chart type."
                self.model.alert_type = "success"
            except Exception as e:
                self.model.alert_message = f"Error reading raster: {e}"
                self.model.alert_type = "error"
            finally:
                self.model.loading = False
        else:
            self.model.alert_message = "Select a raster file."
            self.model.alert_type = "info"

    def _on_plot_click(self, widget, event, data):
        self.model.loading = True
        self.plot_output.clear_output()
        self.model.alert_message = f"Generating {self.model.selected_chart} plot..."
        self.model.alert_type = "info"
        try:
            data1, data2 = get_band_data(
                self.model.input_raster,
                self.model.selected_cov_band,
                self.model.selected_nbr_band,
                enable_mask=self.model.mask_enabled,
                value_to_mask=self.model.mask_value
            )
            self._generate_plot(data1, data2, self.model.selected_chart)
            self.model.alert_message = "Plot generated."
            self.model.alert_type = "success"
        except Exception as e:
            self.model.alert_message = f"Error generating plot: {e}"
            self.model.alert_type = "error"
            traceback.print_exc()
        finally:
            self.model.loading = False

    def _generate_plot(self, data_x, data_y, chart_type):
        with self.plot_output:
            plt.style.use('seaborn-v0_8-whitegrid')
            fig, ax = plt.subplots(figsize=(8, 6))
            nbr_band_text = next((b['text'] for b in self.model.available_bands if b['value'] == self.model.selected_nbr_band), 'NBR (Y)')
            cov_band_text = next((b['text'] for b in self.model.available_bands if b['value'] == self.model.selected_cov_band), 'Covariate (X)')
            title = f"{nbr_band_text} vs. {cov_band_text}"

            if chart_type == 'scatter':
                slope, intercept, r_value, p_value, std_err = stats.linregress(data_x, data_y)
                r_squared = r_value**2
                ax.scatter(data_x, data_y, s=10, alpha=0.5, edgecolors='none', label='Pixel Samples')
                line_x = np.array(ax.get_xlim())
                line_y = intercept + slope * line_x
                ax.plot(line_x, line_y, color='red', linestyle='--', label=f'Fit (R²={r_squared:.3f})')
                ax.set_xlabel(cov_band_text); ax.set_ylabel(nbr_band_text)
                ax.set_title(f"Scatter Plot & Regression: {title}")
                ax.grid(True, which='both', linestyle='--', linewidth=0.5)
                ax.legend()

            elif chart_type == 'violin':
                df = pd.DataFrame({'Covariate': data_x, 'NBR': data_y})
                try:
                    bins = pd.qcut(df['Covariate'], q=6, retbins=True, duplicates='drop')[1]
                    labels = [f'{bins[i]:.1f}-{bins[i+1]:.1f}' for i in range(len(bins)-1)]
                    df['Covariate Bins'] = pd.cut(df['Covariate'], bins=bins, labels=labels, include_lowest=True)
                except Exception:
                     print("Warning: Could not create 6 quantile bins. Trying 4 equal-width bins.")
                     df['Covariate Bins'] = pd.cut(df['Covariate'], bins=4, include_lowest=True)

                # --- UPDATED: Violin plot call ---
                sns.violinplot(
                    x='Covariate Bins',
                    y='NBR',
                    data=df,
                    ax=ax,
                    inner='quartile',
                    palette='viridis',
                    hue='Covariate Bins', # Use x for hue
                    legend=False          # Disable legend
                )
                # --- END UPDATED ---
                ax.set_xlabel(f"Binned {cov_band_text}"); ax.set_ylabel(nbr_band_text)
                ax.set_title(f"Violin Plot: {title}")
                plt.xticks(rotation=30, ha='right')

            plt.tight_layout()
            plt.show()

# --- Initialisation & Display ---
print("Initializing application...")
burn_model = BurnModel()
burn_controller = BurnController(model=burn_model)
print("Displaying application UI...")
burn_controller.app